# Tutorial 1: Download and Analyze CoREB Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hq-bench/coreb/blob/main/notebooks/01_download_and_analyze_data.ipynb)

**CoREB** (Code Retrieval and Reranking Benchmark) is a graded-relevance benchmark for evaluating code retrieval and reranking models across three tasks:

| Task | Query | Target |
|------|-------|--------|
| **Text-to-Code** (T2C) | Natural language description | Code solution |
| **Code-to-Code** (C2C) | Code in language A | Equivalent code in language B |
| **Code-to-Text** (C2T) | Code snippet | Problem description |

In this notebook, you will learn how to:
1. Install the package and download the dataset from HuggingFace
2. Explore the corpus, queries, and relevance judgments (qrels)
3. Analyze data statistics and distributions

**Reference:** Xue et al., *Beyond Retrieval: A Multitask Benchmark and Model for Code Search*, 2025. [arXiv:2605.04615](https://arxiv.org/abs/2605.04615)

## 1. Installation

In [ ]:
!pip install -q coreb datasets

## 2. Download the Dataset from HuggingFace

CoREB is hosted on HuggingFace at [`hq-bench/coreb`](https://huggingface.co/datasets/hq-bench/coreb). It has **8 configs** and **2 splits** (`release_v2602` for training, `release_v2603` for testing).

We'll load the **v202603 (test)** split.

In [ ]:
from datasets import load_dataset

SPLIT = "release_v2603"  # test split (v202602 is the training split)

# Load corpora
code_corpus = load_dataset("hq-bench/coreb", "code_corpus", split=SPLIT)
text_corpus = load_dataset("hq-bench/coreb", "text_corpus", split=SPLIT)

# Load Text-to-Code task
t2c_queries = load_dataset("hq-bench/coreb", "text2code_queries", split=SPLIT)
t2c_qrels   = load_dataset("hq-bench/coreb", "text2code_qrels",   split=SPLIT)

# Load Code-to-Code task
c2c_queries = load_dataset("hq-bench/coreb", "code2code_queries", split=SPLIT)
c2c_qrels   = load_dataset("hq-bench/coreb", "code2code_qrels",   split=SPLIT)

# Load Code-to-Text task
c2t_queries = load_dataset("hq-bench/coreb", "code2text_queries", split=SPLIT)
c2t_qrels   = load_dataset("hq-bench/coreb", "code2text_qrels",   split=SPLIT)

print(f"Code corpus:  {len(code_corpus):,} documents")
print(f"Text corpus:  {len(text_corpus):,} documents")
print(f"T2C queries:  {len(t2c_queries):,}  |  qrels: {len(t2c_qrels):,}")
print(f"C2C queries:  {len(c2c_queries):,}  |  qrels: {len(c2c_qrels):,}")
print(f"C2T queries:  {len(c2t_queries):,}  |  qrels: {len(c2t_qrels):,}")

## 3. Explore the Code Corpus

Each document in the code corpus contains a code solution with metadata about its language and generator model.

In [ ]:
# Inspect columns
print("Code corpus columns:", code_corpus.column_names)
print()

# Look at one example
sample = code_corpus[0]
print(f"code_id:    {sample['code_id']}")
print(f"language:   {sample['language']}")
print(f"problem_id: {sample.get('problem_id', 'N/A')}")
print(f"\nCode snippet (first 500 chars):\n{sample['code'][:500]}")

In [ ]:
from collections import Counter

# Language distribution in the code corpus
lang_counts = Counter(code_corpus["language"])
print("Language distribution in code corpus:")
for lang, count in lang_counts.most_common():
    print(f"  {lang:10s}: {count:4d}")

## 4. Explore the Text Corpus

The text corpus contains problem descriptions — both original statements and LLM-generated noise variants.

In [ ]:
print("Text corpus columns:", text_corpus.column_names)
print()

sample = text_corpus[0]
print(f"text_id: {sample.get('text_id', sample.get('doc_id', 'N/A'))}")
print(f"\nText (first 500 chars):\n{sample['text'][:500]}")

## 5. Explore Queries and Qrels

### 5.1 Text-to-Code (T2C)

T2C queries are natural language descriptions; qrels link each query to relevant code documents with graded relevance:
- **rel=2**: true positive (correct match)
- **rel=1**: hard negative (same problem, wrong/partial match)
- **rel=0**: irrelevant (unjudged docs are implicitly 0)

In [ ]:
print("T2C query columns:", t2c_queries.column_names)
print("T2C qrel columns: ", t2c_qrels.column_names)
print()

# Show a query
q = t2c_queries[0]
print(f"query_id: {q['query_id']}")
print(f"subtask:  {q.get('subtask', 'N/A')}")
print(f"query:    {q['query'][:300]}...")
print()

# Show matching qrels for this query
qid = q["query_id"]
matching = [r for r in t2c_qrels if r["query_id"] == qid]
print(f"Qrels for query '{qid}':")
for r in matching[:5]:
    print(f"  doc_id={r['doc_id']}, relevance={r['relevance']}")

In [ ]:
# Relevance distribution for T2C
t2c_rel_counts = Counter(t2c_qrels["relevance"])
print("T2C relevance distribution:")
for rel, count in sorted(t2c_rel_counts.items()):
    print(f"  rel={rel}: {count:,}")

### 5.2 Subtask Breakdown

T2C queries span different subtasks (e.g., canonical, full, search). Let's group them.

In [ ]:
if "subtask" in t2c_queries.column_names:
    subtask_counts = Counter(t2c_queries["subtask"])
    print("T2C subtask distribution:")
    for st, count in subtask_counts.most_common():
        print(f"  {st:20s}: {count}")
else:
    print("No 'subtask' column found in T2C queries.")

### 5.3 Code-to-Code (C2C) & Code-to-Text (C2T)

In [ ]:
# C2C overview
print("=== Code-to-Code ===")
print(f"Queries: {len(c2c_queries):,}  |  Qrels: {len(c2c_qrels):,}")
c2c_rel = Counter(c2c_qrels["relevance"])
for rel, count in sorted(c2c_rel.items()):
    print(f"  rel={rel}: {count:,}")

print()

# C2T overview
print("=== Code-to-Text ===")
print(f"Queries: {len(c2t_queries):,}  |  Qrels: {len(c2t_qrels):,}")
c2t_rel = Counter(c2t_qrels["relevance"])
for rel, count in sorted(c2t_rel.items()):
    print(f"  rel={rel}: {count:,}")

## 6. Summary Statistics

In [ ]:
print(f"{'':=<60}")
print(f"CoREB v202603 (test split) Summary")
print(f"{'':=<60}")
print(f"{'Code corpus':25s} {len(code_corpus):>8,} documents")
print(f"{'Text corpus':25s} {len(text_corpus):>8,} documents")
print(f"{'':─<60}")
print(f"{'T2C queries':25s} {len(t2c_queries):>8,}")
print(f"{'T2C qrels':25s} {len(t2c_qrels):>8,}  (pos: {t2c_rel_counts.get(2,0):,}, hard-neg: {t2c_rel_counts.get(1,0):,})")
print(f"{'C2C queries':25s} {len(c2c_queries):>8,}")
print(f"{'C2C qrels':25s} {len(c2c_qrels):>8,}  (pos: {c2c_rel.get(2,0):,}, hard-neg: {c2c_rel.get(1,0):,})")
print(f"{'C2T queries':25s} {len(c2t_queries):>8,}")
print(f"{'C2T qrels':25s} {len(c2t_qrels):>8,}  (pos: {c2t_rel.get(2,0):,}, hard-neg: {c2t_rel.get(1,0):,})")
print(f"{'':=<60}")
print(f"Languages: {', '.join(sorted(lang_counts.keys()))}")

## 7. Converting to Evaluation Format

CoREB provides helper functions to convert HuggingFace datasets into the CoIR-compatible dict format used by the evaluator. This is the bridge to Tutorial 2.

In [ ]:
from coreb_runner.benchmark import (
    convert_corpus_to_coir_format,
    convert_queries_to_coir_format,
    convert_qrels_to_coir_format,
)

# Convert T2C data to evaluation format
corpus  = convert_corpus_to_coir_format(list(code_corpus))
queries = convert_queries_to_coir_format(list(t2c_queries))
qrels   = convert_qrels_to_coir_format(list(t2c_qrels))

print(f"Corpus dict:  {len(corpus):,} entries")
print(f"Queries dict: {len(queries):,} entries")
print(f"Qrels dict:   {len(qrels):,} query groups")

# Peek at one entry
first_qid = next(iter(queries))
print(f"\nSample query '{first_qid}': {queries[first_qid][:200]}...")

## Citation

If you use CoREB in your research, please cite:

```bibtex
@article{xue2025coreb,
  title   = {Beyond Retrieval: A Multitask Benchmark and Model for Code Search},
  author  = {Xue, Siqiao and Liao, Zihan and Qin, Jin and Zhang, Ziyin and Mu, Yixiang and Zhou, Fan and Yu, Hang},
  journal = {arXiv preprint arXiv:2605.04615},
  year    = {2025},
  url     = {https://arxiv.org/abs/2605.04615}
}
```